# Gene ↔ Gene Relation-Wise Merge

Merges Gene–Gene triples from Monarch, DRKG, PrimeKG, PharmKG, Hetionet, GPKG, TARKG,
iBKH, Harmonizome (×8), and hald; resolves missing head/tail gene names via NCBI;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd
import re

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_GeneENTITY/ALL_GENE_GeneENTITY.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries — NCBI and ENSEMBL

In [2]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### Monarch

In [3]:
Monarch_Gene_Gene = pd.read_csv(PROC_DIR + 'Monarch/Human/GENE_GENE_HUMAN_HUMAN_MONARCH.csv')
Monarch_Gene_Gene.columns = Monarch_Gene_Gene.columns.str.lower()
Monarch_Gene_Gene = Monarch_Gene_Gene.drop(columns=['tail']).rename(columns={'tail.1': 'tail'})
# Monarch_Gene_Gene.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
print(f"Monarch: {len(Monarch_Gene_Gene):,} rows")

Monarch_Gene_Gene['kg_type'] = 'Generalised'
Monarch_Gene_Gene['kg_source'] = 'Monarch_KG'
Monarch_Gene_Gene['species'] = 'Homo species'
Monarch_Gene_Gene.head(2)

Monarch: 1,324,580 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,tail_name,head_orignal,species_species,kg_type,species
0,ISG15,Gene_Gene,RPS15,Gene,interacts_with,Gene,infores:string,Monarch_KG,ISG15 ubiquitin like modifier,ribosomal protein S15,Homo sapiens,Homo sapiens,NCBI_ID,NCBI_ID,ISG15,RPS15,HGNC:4053,Homo sapiens_Homo sapiens,Generalised,Homo species
1,ISG15,Gene_Gene,SAMHD1,Gene,interacts_with,Gene,infores:string,Monarch_KG,ISG15 ubiquitin like modifier,SAM and HD domain containing deoxynucleoside t...,Homo sapiens,Homo sapiens,NCBI_ID,NCBI_ID,ISG15,SAMHD1,HGNC:4053,Homo sapiens_Homo sapiens,Generalised,Homo species


In [4]:
Monarch_Gene_Gene.columns

Index(['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
       'relation_source', 'kg_source', 'head_detail_name', 'tail_detail_name',
       'head_taxon_name', 'tail_taxon_name', 'head_id_is', 'tail_id_is',
       'head_name', 'tail_name', 'head_orignal', 'species_species', 'kg_type',
       'species'],
      dtype='object')

### DRKG

In [5]:
DRKG_Gene_Gene = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Gene_Gene.csv')
DRKG_Gene_Gene.columns = DRKG_Gene_Gene.columns.str.lower()
print(f"DRKG: {len(DRKG_Gene_Gene):,} rows")
DRKG_Gene_Gene['kg_type'] = 'Generalised'
DRKG_Gene_Gene['kg_source'] = 'DRKG'
DRKG_Gene_Gene['species'] = 'Homo species'
DRKG_Gene_Gene.head(2)

DRKG: 2,326,174 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_name,head_detail_name,head_ens,tail_detail_name,tail_ens,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,F8,Gene_Gene,F8,Gene,bioarx::HumGenHumGen:Gene:Gene,Gene,DRKG,2157,coagulation factor VIII,ENSG00000185010,coagulation factor VIII,ENSG00000185010,NCBI_ID,NCBI_ID,Gene::2157,Gene::2157,Generalised,Homo species
1,F8,Gene_Gene,PHYH,Gene,bioarx::HumGenHumGen:Gene:Gene,Gene,DRKG,2157,coagulation factor VIII,ENSG00000185010,phytanoyl-CoA 2-hydroxylase,ENSG00000107537,NCBI_ID,NCBI_ID,Gene::2157,Gene::5264,Generalised,Homo species


### PrimeKG

In [6]:
PrimeKG_Gene_Gene = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Gene_Gene.csv')
PrimeKG_Gene_Gene.columns = PrimeKG_Gene_Gene.columns.str.lower()
print(f"PrimeKG: {len(PrimeKG_Gene_Gene):,} rows")
PrimeKG_Gene_Gene['kg_type'] = 'Generalised'
PrimeKG_Gene_Gene['kg_source'] = 'PrimeKG'
PrimeKG_Gene_Gene['species'] = 'Homo species'
PrimeKG_Gene_Gene.head(2)

PrimeKG: 321,075 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_alias_mapped,head_detail_name,head_ens,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,KIF15,Gene_Gene,PHYHIP,Gene,ppi,Gene,PrimeKG,NCBI_Symbol,NCBI_Symbol,KIF15,kinesin family member 15,ENSG00000163808,PHYHIP,phytanoyl-CoA 2-hydroxylase interacting protein,ENSG00000168490,Generalised,Homo species
1,PNMA1,Gene_Gene,GPANK1,Gene,ppi,Gene,PrimeKG,NCBI_Symbol,NCBI_Symbol,PNMA1,PNMA family member 1,ENSG00000176903,GPANK1,G-patch domain and ankyrin repeats 1,ENSG00000204438,Generalised,Homo species


### PharmKG

In [7]:
PharmKG_Gene_Gene = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Gene_Gene.csv')
PharmKG_Gene_Gene.columns = PharmKG_Gene_Gene.columns.str.lower()
print(f"PharmKG: {len(PharmKG_Gene_Gene):,} rows")
PharmKG_Gene_Gene['kg_type'] = 'Generalised'
PharmKG_Gene_Gene['kg_source'] = 'PharmKG'
PharmKG_Gene_Gene['species'] = 'Homo species'
PharmKG_Gene_Gene.head(2)

PharmKG: 102,072 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,tail_alias_mapped,head_id_is,tail_id_is,pubmed_id,sentence_tokenized,kg_type,species
0,ACTN4,Gene_Gene,IGF2,gene,V+,gene,PharmKG,actinin alpha 4,insulin like growth factor 2,IGF2,NCBI_ID,NCBI_ID,19055724.0,"'IGF2 , ACTN4 , AXL , and SHC1 were among the ...",Generalised,Homo species
1,ADORA3,Gene_Gene,ADORA3,gene,H,gene,PharmKG,adenosine A3 receptor,adenosine A3 receptor,ADORA3,NCBI_ID,NCBI_ID,"22028334.0, 22028334.0, 22028334.0, 22028334.0",'The increased phagocytosis was attenuated by ...,Generalised,Homo species


### Hetionet

In [8]:
Hetionet_Gene_Gene = pd.read_csv(PROC_DIR + 'Hetionet/Gene_Gene_Hetionet.csv')
Hetionet_Gene_Gene.columns = Hetionet_Gene_Gene.columns.str.lower()
print(f"Hetionet: {len(Hetionet_Gene_Gene):,} rows")
Hetionet_Gene_Gene['kg_type'] = 'Generalised'
Hetionet_Gene_Gene['kg_source'] = 'Hetionet'
Hetionet_Gene_Gene['species'] = 'Homo species'
Hetionet_Gene_Gene.head(2)

Hetionet: 474,415 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_name,head_detail_name,tail_name,tail_detail_name,kg_type,species
0,CALM1,Gene_Gene,VHL,Gene,Gene–interacts–Gene,Gene,Hetionet,NCBI_ID,NCBI_ID,801,calmodulin 1,7428,von Hippel-Lindau tumor suppressor,Generalised,Homo species
1,TRIM27,Gene_Gene,MED21,Gene,Gene–interacts–Gene,Gene,Hetionet,NCBI_ID,NCBI_ID,5987,tripartite motif containing 27,9412,mediator complex subunit 21,Generalised,Homo species


### GPKG

In [9]:
GPKG_Gene_Gene = pd.read_csv(PROC_DIR + 'GPKG/Gene_Gene_GPKG.csv')
GPKG_Gene_Gene.columns = GPKG_Gene_Gene.columns.str.lower()
print(f"GPKG: {len(GPKG_Gene_Gene):,} rows")
GPKG_Gene_Gene['kg_type'] = 'Generalised'
GPKG_Gene_Gene['kg_source'] = 'GPKG'
GPKG_Gene_Gene['species'] = 'Homo species'
GPKG_Gene_Gene.head(2)

GPKG: 1,838,423 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_orignal,tail_orignal,head_alias_mapped,head_detail_name,tail_alias_mapped,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,LCN10,Gene_Gene,LCN6,Gene,coexpresses,Gene,GPKG,ENSG00000187922,ENSG00000267206,LCN10,lipocalin 10,LCN6,lipocalin 6,NCBI_ID,NCBI_ID,Generalised,Homo species
1,RPL31,Gene_Gene,RPL32,Gene,coexpresses,Gene,GPKG,ENSG00000071082,ENSG00000144713,RPL31,ribosomal protein L31,RPL32,ribosomal protein L32,NCBI_ID,NCBI_ID,Generalised,Homo species


### TARKG

In [10]:
TARKG_Gene_Gene = pd.read_csv(PROC_DIR + 'TARKG/Gene_Gene_1_TARKG.csv')
TARKG_Gene_Gene.columns = TARKG_Gene_Gene.columns.str.lower()
print(f"TARKG: {len(TARKG_Gene_Gene):,} rows")
TARKG_Gene_Gene['kg_type'] = 'Generalised'
TARKG_Gene_Gene['kg_source'] = 'TARKG'
TARKG_Gene_Gene['species'] = 'Homo species'
TARKG_Gene_Gene.head(2)

TARKG: 6,016,029 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,NUP62,Gene_Gene,TOP2A,Gene,GENE_CATALYSIS_GENE,Gene,TARKG,nucleoporin 62,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,OpenBioLink-1528454,OpenBioLink,0,Generalised,Homo species
1,NUP62,Gene_Gene,TOP2A,Gene,STRING::CATALYSIS::Gene:Gene,Gene,TARKG,nucleoporin 62,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,DRKG-5143357,DRKG,0,Generalised,Homo species


### iBKH

In [11]:
iBKH_Gene_Gene = pd.read_csv(PROC_DIR + 'iBKH/Gene_Gene_ibkh.csv')
iBKH_Gene_Gene.columns = iBKH_Gene_Gene.columns.str.lower()
print(f"iBKH: {len(iBKH_Gene_Gene):,} rows")
iBKH_Gene_Gene['kg_type'] = 'Generalised'
iBKH_Gene_Gene['kg_source'] = 'iBKH'
iBKH_Gene_Gene['species'] = 'Homo species'
iBKH_Gene_Gene.head(2)

iBKH: 711,649 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,IMP3,Gene_Gene,OR8U8,Gene,Gene,iBKH,IMP U3 small nucleolar ribonucleoprotein 3,olfactory receptor family 8 subfamily U member 8,NCBI_ID,NCBI_ID,Generalised,Homo species
1,FADD,Gene_Gene,C1orf56,Gene,Gene,iBKH,Fas associated via death domain,chromosome 1 open reading frame 56,NCBI_ID,NCBI_ID,Generalised,Homo species


### Harmonizome

In [12]:
Harmonizome_Gene_Gene = pd.read_csv(PROC_DIR + 'harmonizome/Gene_Gene_Harmonizome.csv')
Harmonizome_Gene_Gene.columns = Harmonizome_Gene_Gene.columns.str.lower()
print(f"Harmonizome: {len(Harmonizome_Gene_Gene):,} rows")

Harmonizome_Gene_Gene['kg_type'] = 'Generalised'
Harmonizome_Gene_Gene['kg_source'] = 'Harmonizome'
Harmonizome_Gene_Gene['species'] = 'Homo species'
Harmonizome_Gene_Gene.head(2)

Harmonizome: 2,750,021 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,WWOX,Gene_Gene,MYH16,Gene,Gene,BioGRID Protein-Protein Interactions,Harmonizome,WW domain containing oxidoreductase,myosin heavy chain 16,NCBI_ID,NCBI_ID,Generalised,Homo species
1,WWOX,Gene_Gene,RPS15AP21,Gene,Gene,BioGRID Protein-Protein Interactions,Harmonizome,WW domain containing oxidoreductase,ribosomal protein S15a pseudogene 21,NCBI_ID,NCBI_ID,Generalised,Homo species


### hald

In [13]:
hald = pd.read_csv(PROC_DIR + 'hald/Gene_Gene_HALD.csv')
# hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows")
hald['kg_type'] = 'Aging'
hald['kg_source'] = 'HALD_KG'
hald['species'] = 'Homo species'
hald.head(2)

hald: 4,266 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,GHRH,Gene_Gene,GGH,Gene,elevate,Gene,HALD_KG,growth hormone releasing hormone,gamma-glutamyl hydrolase,NCBI_ID,NCBI_ID,Aging,Homo species
1,CD8A,Gene_Gene,CD4,Gene,considered,Gene,HALD_KG,CD8 subunit alpha,CD4 molecule,NCBI_ID,NCBI_ID,Aging,Homo species


### Pheknowlator

In [14]:
pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Gene_Gene.csv')
# pheknowlator.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
pheknowlator.columns = pheknowlator.columns.str.lower()
print(f"pheknowlator: {len(pheknowlator):,} rows")
pheknowlator['kg_type'] = 'Generalised'
pheknowlator['kg_source'] = 'pheknowlator'
pheknowlator['species'] = 'Homo species'
pheknowlator['head_id_is'] = 'NCBI_ID'
pheknowlator['tail_id_is'] = 'NCBI_ID'
pheknowlator.head(2)

pheknowlator: 410,698 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,kg_source,kg_type,species,head_id_is,tail_id_is
0,PTGER3,Gene_Gene,CXCL13,Gene,Gene,prostaglandin E receptor 3,C-X-C motif chemokine ligand 13,pheknowlator,Generalised,Homo species,NCBI_ID,NCBI_ID
1,DYNC1H1,Gene_Gene,PDS5B,Gene,Gene,dynein cytoplasmic 1 heavy chain 1,PDS5 cohesin associated factor B,pheknowlator,Generalised,Homo species,NCBI_ID,NCBI_ID


## 3. Check and Fix Duplicate Columns

In [15]:
SOURCE_DFS = [
    ('Monarch_Gene_Gene',  Monarch_Gene_Gene),
    ('DRKG_Gene_Gene',     DRKG_Gene_Gene),
    ('PrimeKG_Gene_Gene',  PrimeKG_Gene_Gene),
    ('PharmKG_Gene_Gene',  PharmKG_Gene_Gene),
    ('Hetionet_Gene_Gene', Hetionet_Gene_Gene),
    ('GPKG_Gene_Gene',     GPKG_Gene_Gene),
    ('TARKG_Gene_Gene',    TARKG_Gene_Gene),
    ('iBKH_Gene_Gene',     iBKH_Gene_Gene),
    ('Harmonizome_Gene_Gene',     Harmonizome_Gene_Gene),
    ('hald', hald),
    ('pheknowlator',     pheknowlator),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Gene_Gene] ✓ no duplicates
[DRKG_Gene_Gene] ✓ no duplicates
[PrimeKG_Gene_Gene] ✓ no duplicates
[PharmKG_Gene_Gene] ✓ no duplicates
[Hetionet_Gene_Gene] ✓ no duplicates
[GPKG_Gene_Gene] ✓ no duplicates
[TARKG_Gene_Gene] ✓ no duplicates
[iBKH_Gene_Gene] ✓ no duplicates
[Harmonizome_Gene_Gene] ✓ no duplicates
[hald] ✓ no duplicates
[pheknowlator] ✓ no duplicates


In [16]:
SOURCE_DFS = [(name, df.loc[:, ~df.columns.duplicated(keep='first')]) for name, df in SOURCE_DFS]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[Monarch_Gene_Gene] ✓ clean
[DRKG_Gene_Gene] ✓ clean
[PrimeKG_Gene_Gene] ✓ clean
[PharmKG_Gene_Gene] ✓ clean
[Hetionet_Gene_Gene] ✓ clean
[GPKG_Gene_Gene] ✓ clean
[TARKG_Gene_Gene] ✓ clean
[iBKH_Gene_Gene] ✓ clean
[Harmonizome_Gene_Gene] ✓ clean
[hald] ✓ clean
[pheknowlator] ✓ clean


## 4. Align Schemas and Concatenate

In [17]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 16,279,402 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,ISG15,Gene_Gene,RPS15,Gene,interacts_with,Gene,Monarch_KG,NCBI_ID,NCBI_ID,ISG15 ubiquitin like modifier,ribosomal protein S15,Generalised,Homo species
1,ISG15,Gene_Gene,SAMHD1,Gene,interacts_with,Gene,Monarch_KG,NCBI_ID,NCBI_ID,ISG15 ubiquitin like modifier,SAM and HD domain containing deoxynucleoside t...,Generalised,Homo species
2,ISG15,Gene_Gene,MX1,Gene,interacts_with,Gene,Monarch_KG,NCBI_ID,NCBI_ID,ISG15 ubiquitin like modifier,MX dynamin like GTPase 1,Generalised,Homo species
3,ISG15,Gene_Gene,IFNA4,Gene,interacts_with,Gene,Monarch_KG,NCBI_ID,NCBI_ID,ISG15 ubiquitin like modifier,interferon alpha 4,Generalised,Homo species
4,ISG15,Gene_Gene,GAPDH,Gene,interacts_with,Gene,Monarch_KG,NCBI_ID,NCBI_ID,ISG15 ubiquitin like modifier,glyceraldehyde 3 phosphate dehydrogenase pseud...,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16279397,KLHL11,Gene_Gene,RPS27A,Gene,NaN,Gene,pheknowlator,NCBI_ID,NCBI_ID,kelch like family member 11,ribosomal protein S27a,Generalised,Homo species
16279398,PES1,Gene_Gene,RPL23,Gene,NaN,Gene,pheknowlator,NCBI_ID,NCBI_ID,pescadillo ribosomal biogenesis factor 1,ribosomal protein L23,Generalised,Homo species
16279399,FLT3,Gene_Gene,STAT3,Gene,NaN,Gene,pheknowlator,NCBI_ID,NCBI_ID,fms related receptor tyrosine kinase 3,signal transducer and activator of transcripti...,Generalised,Homo species
16279400,KLHL9,Gene_Gene,ASB15,Gene,NaN,Gene,pheknowlator,NCBI_ID,NCBI_ID,kelch like family member 9,ankyrin repeat and SOCS box containing 15,Generalised,Homo species


In [18]:
# final_df[final_df['

## 5. Standardise Fixed-Value Columns

In [19]:
final_df['head_type']  = 'Gene'
final_df['tail_type']  = 'Gene'
final_df['relation']   = 'Gene_Gene'
final_df['head_id_is'] = 'NCBI_ID'
final_df['tail_id_is'] = 'NCBI_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Gene_Gene'}
Unique kg_source: {'GPKG', 'iBKH', 'pheknowlator', 'Harmonizome', 'PharmKG', 'PrimeKG', 'DRKG', 'HALD_KG', 'Hetionet', 'TARKG', 'Monarch_KG'}


## 6. Deduplicate

In [20]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 7,127,445 rows


## 7. QC — NaN Check

In [21]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,2170388,0,2170388
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,682,0,682


## 8. Repair Missing Gene Names (head and tail)

For rows missing a name: strip `-`, map synonyms → canonical symbol, then NCBI Symbol → description.

In [22]:
for node in ['head', 'tail']:
    mask = final_df_dedup[f'{node}_detail_name'].isna()
    print(f"{node}: rows needing repair: {mask.sum():,}")
    final_df_dedup.loc[mask, node] = final_df_dedup.loc[mask, node].str.replace('-', '', regex=False)
    final_df_dedup.loc[mask, node] = (
        final_df_dedup.loc[mask, node]
        .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
        .fillna(final_df_dedup.loc[mask, node])
    )
    mask2 = final_df_dedup[f'{node}_detail_name'].isna()
    final_df_dedup.loc[mask2, f'{node}_detail_name'] = final_df_dedup.loc[mask2, node].map(NCBI_ALL_Symb_Desc_dict)
    print(f"{node}: still missing: {final_df_dedup[f'{node}_detail_name'].isna().sum():,}")

head: rows needing repair: 682
head: still missing: 0
tail: rows needing repair: 1,013
tail: still missing: 0


In [23]:
final_df_dedup[final_df_dedup['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


In [24]:
final_df_dedup[final_df_dedup['head_detail_name'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 9. Add Schema Columns and Enforce Column Order

In [25]:
final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (7127445, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,3DL2,Gene_Gene,KIR2DL4,Gene,V+,Gene,PharmKG,NCBI_ID,NCBI_ID,"killer cell immunoglobulin like receptor, thre...","killer cell immunoglobulin like receptor, two ...",Generalised,Homo species
1,6PGD,Gene_Gene,ME1,Gene,B,Gene,PharmKG,NCBI_ID,NCBI_ID,phosphogluconate dehydrogenase,protocadherin beta 16,Generalised,Homo species
2,A1,Gene_Gene,VWF,Gene,Q,Gene,PharmKG,NCBI_ID,NCBI_ID,immunoglobulin kappa variable 2D-30,von Willebrand factor,Generalised,Homo species
3,A13,Gene_Gene,BRS3,Gene,Q,Gene,PharmKG,NCBI_ID,NCBI_ID,immunoglobulin kappa variable 2D-18 (pseudogene),bombesin receptor subtype 3,Generalised,Homo species
4,A18,Gene_Gene,TXN,Gene,Rg,Gene,PharmKG,NCBI_ID,NCBI_ID,immunoglobulin kappa variable 2-29,thioredoxin 2,Generalised,Homo species


In [28]:
#

## 10. Save Output

In [27]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 7,127,445 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_GeneENTITY/ALL_GENE_GeneENTITY.parquet
